# 03 Terrain and capacity

Tests whether the master plan's stated comfortable carrying capacity (CCC) can be reproduced from its own stated per-lift inputs (claim C025), using the formula the appendix itself describes (claim C024). This notebook does not attempt the LiDAR slope-by-ability-class analysis in the step's original question; see Open issues in steps/03 for why.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"
rounding_tolerance = 2

## Per-lift table (transcribed from Appendix A)

The rows below are transcribed from data/raw/S011_rmr_appendix_a_ccc_tables.pdf, Tables 1-1 to 1-4, via pdfplumber's table extraction (pypdf's plain-text extraction jumbled these tables' row order, so pdfplumber's structured extraction was used instead and checked by hand against the same tables' raw text). Kept as a literal transcription rather than re-parsing the PDF here, same reasoning as notebook 02: one place does the parsing, everything else reads the result.

In [ ]:
LIFT_ROWS = """
phase,map_ref,slope_length_m,vert_rise_m,hourly_capacity,oper_hours,access_role_pct,misload_pct,stated_adjusted_hourly,stated_vtm_day_000,weighted_vertical_demand_m,stated_ccc
Phase 1 (Existing),Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Phase 1 (Existing),Turtle Creek Carpet,176,30.5,1800,7,0,2,1764,1000,1000,377
Phase 1 (Existing),2,1011,280,2185,7,85,10,109,0,1583,135
Phase 1 (Existing),5,2391,900,2357,7,75,10,353,879,3394,656
Phase 1 (Existing),12,1902,633,2200,6.5,5,5,1980,6829,3379,2411
Phase 1 (Existing),14,1880,530,1800,6,0,5,1710,6473,5857,928
Phase 2,Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Phase 2,Alpine Carpet,250,50,1800,7,85,2,234,1000,1000,82
Phase 2,2,1011,280,2800,7,85,10,140,0,1583,173
Phase 2,5,2391,900,2800,7,75,10,420,879,3394,780
Phase 2,12,1902,633,2600,6.5,5,5,2340,6829,3379,2849
Phase 2,14,1880,530,2600,6,0,5,2470,6473,5857,1341
Phase 2,3,1006,280,2400,7,50,10,960,6268,4350,433
Phase 2,6,1983,658,2400,7,30,10,1440,8900,7150,928
Phase 2,11,438,125,1800,7,5,10,1530,1342,3500,383
Phase 2,13,1216,412,1800,6.5,0,10,1620,1715,9625,451
Phase 2,15,2111,885,2400,6,0,10,2160,9232,8625,1330
Phase 2,18,1235,444,2400,6.5,0,10,2160,7322,8400,742
Phase 3,Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Phase 3,Alpine Carpet,250,50,1800,7,85,2,234,1000,1000,82
Phase 3,2,1011,280,2800,7,85,10,140,0,1583,173
Phase 3,5,2391,900,2800,7,75,10,420,879,3394,780
Phase 3,12,1902,633,2600,6.5,5,5,2340,6829,3379,2849
Phase 3,14,1880,530,2600,6,0,5,2470,6473,5857,1341
Phase 3,3,1006,280,2400,7,50,10,960,6268,4350,433
Phase 3,6,1983,658,2400,7,30,10,1440,8900,7150,928
Phase 3,11,438,125,1800,7,5,10,1530,1342,3500,383
Phase 3,13,1216,412,1800,6.5,0,10,1620,1715,9625,451
Phase 3,15,2111,885,2400,6,0,10,2160,9232,8625,1330
Phase 3,18,1235,444,2400,6.5,0,10,2160,7322,8400,742
Phase 3,19,1193,397,1500,6,0,5,1425,3392,7934,428
Phase 3,21,513,63,1800,6.5,100,0,0,0,0,0
Phase 3,22,1225,500,2400,6.8,15,5,1920,6527,4423,1476
Phase 3,23,189,21,1000,6,0,10,900,116,1000,113
Phase 3,24,120,9,500,6,0,5,475,26,1000,26
Phase 3,25,1093,451,2400,6,100,0,0,0,0,0
Buildout,Lil' Bit,103,15.45,1500,7,75,2,345,1000,1000,37
Buildout,Alpine Carpet,250,50,1800,7,85,2,234,1000,1000,82
Buildout,2,1011,280,2800,7,85,10,140,0,1583,173
Buildout,5,2391,900,2800,7,75,10,420,879,3394,780
Buildout,12,1902,633,2600,6.5,5,5,2340,6829,3379,2849
Buildout,14,1880,530,2600,6,0,5,2470,6473,5857,1341
Buildout,3,1006,280,2400,7,50,10,960,6268,4350,433
Buildout,6,1983,658,2400,7,30,10,1440,8900,7150,928
Buildout,11,438,125,1800,7,5,10,1530,1342,3500,383
Buildout,13,1216,412,1800,6.5,0,10,1620,1715,9625,451
Buildout,15,2111,885,2400,6,0,10,2160,9232,8625,1330
Buildout,18,1235,444,2400,6.5,0,10,2160,7322,8400,742
Buildout,19,1193,397,1500,6,0,5,1425,3392,7934,428
Buildout,21,513,63,1800,6.5,100,0,0,0,0,0
Buildout,22,1225,500,2400,6.8,15,5,1920,6527,4423,1476
Buildout,23,189,21,1000,6,0,10,900,116,1000,113
Buildout,24,120,9,500,6,0,5,475,26,1000,26
Buildout,25,1093,451,2400,6,100,0,0,0,0,0
Buildout,1,1198,382,3000,7,75,10,450,1203,2500,481
Buildout,4,1160,399,2800,7,85,10,140,391,5000,78
Buildout,7,1358,421,2800,7,5,5,2520,7420,5166,1438
Buildout,8,976,164,1800,7,100,0,0,0,0,0
Buildout,9,701,94,1800,6.5,0,15,1530,931,2500,374
Buildout,10,1559,414,2400,6.8,10,5,2040,5746,4560,1259
Buildout,16,1313,395,1800,6.5,50,10,720,1847,8500,217
Buildout,17,1993,633,2400,6.5,5,5,2160,8885,5441,1633
Buildout,20,1746,579,2400,6.25,0,5,2280,8253,7403,1115
"""

## Load the ledger

Confirms C024 and C025, the claims behind the formula and the phase totals this notebook tries to reproduce, still point to the same source before using them.

In [ ]:
import io
import sys

import pandas as pd

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C024"]["source_id"] == "S011"
assert ledger["claims"]["C025"]["source_id"] == "S011"

lifts = pd.read_csv(io.StringIO(LIFT_ROWS))
print(f"{len(lifts)} (phase, lift) rows across {lifts['phase'].nunique()} phases")

## Reproduce Adjusted Hourly Capacity and CCC

C024 gives the top-level formula, CCC = Vertical Rise x Hourly Capacity x Operating Hours x Loading Efficiency / Weighted Vertical Demand, but the appendix text does not spell out Loading Efficiency as its own number. Testing arithmetic against the table's own Adjusted Hourly Capacity column shows it equals Hourly Capacity x (1 - Up-Mtn Access Role% - Misload Lift Stop%); CCC then follows from Adjusted Hourly Capacity x Vert Rise x Oper Hours / Weighted Vertical Demand. Neither equation is a sentence written in the source, so this is this notebook's own reverse-engineering of the tables (a judgment call, not a claim about what the text says), checked below against every row and phase total the appendix actually states.

In [ ]:
lifts["computed_adjusted_hourly"] = lifts["hourly_capacity"] * (
    1 - lifts["access_role_pct"] / 100 - lifts["misload_pct"] / 100
)
lifts["computed_ccc"] = (
    lifts["computed_adjusted_hourly"] * lifts["vert_rise_m"] * lifts["oper_hours"]
    / lifts["weighted_vertical_demand_m"].replace(0, pd.NA)
).fillna(0)
lifts["ccc_diff"] = (lifts["computed_ccc"] - lifts["stated_ccc"]).round(1)
lifts[["phase", "map_ref", "stated_ccc", "computed_ccc", "ccc_diff"]]

## Compare phase totals against step 02's output

Reads data/processed/02_ccc_by_phase.csv (written by notebook 02 from the same claim, C025) rather than retyping the four phase totals a second time, so the two notebooks cannot silently drift apart on the same number.

In [ ]:
ccc_by_phase = pd.read_csv(f"{processed_dir}/02_ccc_by_phase.csv")
computed_by_phase = (
    lifts.groupby("phase", sort=False)["computed_ccc"].sum().round().astype(int)
)
phase_order = ["Phase 1 (Existing)", "Phase 2", "Phase 3", "Buildout"]
comparison = ccc_by_phase.set_index("phase").loc[phase_order]
comparison["computed_ccc_skiers"] = computed_by_phase.loc[phase_order].values
comparison["diff"] = comparison["computed_ccc_skiers"] - comparison["ccc_skiers"]
comparison

## Write outputs

The per-lift reproduction and the phase-level comparison, so step 08's synthesis can cite how well the chain's first link actually reproduces, not just its final numbers.

In [ ]:
import os

os.makedirs(processed_dir, exist_ok=True)
lifts.to_csv(f"{processed_dir}/03_lift_ccc_reproduction.csv", index=False, encoding="utf-8")
comparison.to_csv(f"{processed_dir}/03_ccc_phase_reproduction_summary.csv", encoding="utf-8")
print("wrote 03_lift_ccc_reproduction.csv, 03_ccc_phase_reproduction_summary.csv")

## Checks

Every phase total, and every individual lift's CCC, should match the appendix's own stated figure within a small rounding tolerance. A mismatch bigger than that would mean either this notebook's transcription has an error or the reverse-engineered formula is wrong, and either way it should fail loudly rather than report a reproduction that did not actually happen.

In [ ]:
max_lift_diff = lifts["ccc_diff"].abs().max()
max_phase_diff = comparison["diff"].abs().max()
assert max_lift_diff <= rounding_tolerance, f"largest per-lift CCC diff was {max_lift_diff}"
assert max_phase_diff <= rounding_tolerance, f"largest phase-total CCC diff was {max_phase_diff}"
print(f"checks passed: max per-lift diff {max_lift_diff}, max phase-total diff {max_phase_diff}")

# Spatial terrain analysis (steps/03's proposed addition)

Everything below this point uses a real DEM (NRCan HRDEM 1m, claim C051) and real lift/run geometry (OpenStreetMap, C052-C053) instead of relying only on the master plan's self-reported terrain figures (C032). The project CRS is EPSG:26911 (step 01); every other CRS encountered here is reprojected to it explicitly, with a check that units read as metres afterward.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"
raw_dir = "data/raw"
dem_dtm_url = "https://canelevation-dem.s3.ca-central-1.amazonaws.com/hrdem-mosaic-1m/2_4-mosaic-1m-dtm.tif"
project_crs = "EPSG:26911"
dem_clip_buffer_m = 200
slope_threshold_sets = {
    "sac_ski_touring": [30, 35, 40, 45],   # C067, degrees; steeper, not for groomed downhill
    "folk_banding_pct": [25, 40],           # C069, percent grade; gentler, weakly sourced
}
aoi_area_tolerance_pct = 5      # max allowed drift from C032's 1,263 ha before the check fails
lift_vert_rise_tolerance_m = 5  # max allowed drift from a plan map_ref's vert_rise for a "close" match
name_similarity_threshold = 0.5  # difflib ratio above which a name-based match is worth flagging

## Load the ledger

Confirms the claims this spatial method rests on are still pointing at the sources this session actually queried.

In [ ]:
import sys

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
for claim_id, source_id in [("C051", "S048"), ("C052", "S050"), ("C063", "S050")]:
    assert ledger["claims"][claim_id]["source_id"] == source_id
print("spatial claims confirmed against their sources")

## Area of interest

OpenStreetMap's landuse=winter_sports way is the only candidate AOI boundary found (C063); no dedicated ski-area or tenure boundary source exists. Read from the saved Overpass response rather than re-querying, reprojected from OSM's native EPSG:4326 to the project CRS.

In [ ]:
import json

import geopandas as gpd
import pyproj
from shapely.geometry import Polygon
from shapely.ops import transform as shp_transform

with open(f"{raw_dir}/S050_overpass_aoi.json", encoding="utf-8") as f:
    aoi_raw = json.load(f)
aoi_way = aoi_raw["elements"][0]
aoi_4326 = Polygon([(pt["lon"], pt["lat"]) for pt in aoi_way["geometry"]])

to_project_crs = pyproj.Transformer.from_crs("EPSG:4326", project_crs, always_xy=True).transform
aoi_geom = shp_transform(to_project_crs, aoi_4326)
aoi = gpd.GeoDataFrame(
    {"osm_way_id": [aoi_way["id"]], "name": [aoi_way["tags"].get("name")]},
    geometry=[aoi_geom], crs=project_crs,
)
aoi_area_ha = aoi.geometry.area.iloc[0] / 10_000
print(f"AOI area: {aoi_area_ha:.1f} ha, CRS: {aoi.crs}")

## Lift and run geometry

From the same Overpass query used in the Phase 1 access test (saved, not re-queried). Aerialway lines exclude aerialway=station point/marker features (C052's own distinction); piste ways keep their piste:difficulty tag.

In [ ]:
with open(f"{raw_dir}/S050_overpass_lifts_pistes.json", encoding="utf-8") as f:
    lifts_pistes_raw = json.load(f)

lift_elements = [
    e for e in lifts_pistes_raw["elements"]
    if e.get("tags", {}).get("aerialway") and e["tags"]["aerialway"] != "station"
]
run_elements = [
    e for e in lifts_pistes_raw["elements"]
    if e.get("tags", {}).get("piste:type") == "downhill"
]
print(f"{len(lift_elements)} lift lines, {len(run_elements)} runs")

## Reproject lift and run lines to the project CRS

One helper used for both, since both are OSM ways with the same geometry shape (a list of lat/lon points).

In [ ]:
def osm_way_to_line(element):
    from shapely.geometry import LineString

    line_4326 = LineString([(pt["lon"], pt["lat"]) for pt in element["geometry"]])
    return shp_transform(to_project_crs, line_4326)


lifts_gdf = gpd.GeoDataFrame(
    {
        "osm_id": [e["id"] for e in lift_elements],
        "name": [e["tags"].get("name", f"id{e['id']}") for e in lift_elements],
        "aerialway": [e["tags"]["aerialway"] for e in lift_elements],
    },
    geometry=[osm_way_to_line(e) for e in lift_elements], crs=project_crs,
)
runs_gdf = gpd.GeoDataFrame(
    {
        "osm_id": [e["id"] for e in run_elements],
        "name": [e["tags"].get("name") for e in run_elements],
        "osm_difficulty": [e["tags"].get("piste:difficulty") for e in run_elements],
    },
    geometry=[osm_way_to_line(e) for e in run_elements], crs=project_crs,
)
print("lifts and runs reprojected;", lifts_gdf.crs, runs_gdf.crs)

## DEM: windowed clip and reproject

Opens the HRDEM mosaic's dtm asset directly over HTTP (GDAL's /vsicurl/ under rioxarray) and clips to the union of the AOI, lift, and run bounding boxes plus a small buffer, never reading the full national mosaic. Lifts and runs are included in the clip bounds, not just the AOI, because some runs (mostly unnamed freeride ways) extend past the AOI polygon; clipping to the AOI alone left those points outside the window and their DEM-derived stats silently NaN. Computed on the DEM's native EPSG:3979 grid, then reprojected once to the project CRS, checking pixel size afterward.

In [ ]:
import rioxarray

# Union of the AOI, every lift, and every run's bounds, not just the AOI:
# some runs (mostly unnamed freeride ways) extend beyond the AOI polygon,
# and clipping to the AOI alone left those runs' points outside the DEM
# window, producing silent NaN stats for them.
import numpy as np

all_bounds = np.array([aoi.total_bounds, lifts_gdf.total_bounds, runs_gdf.total_bounds])
combined_bounds_project_crs = (
    all_bounds[:, 0].min(), all_bounds[:, 1].min(), all_bounds[:, 2].max(), all_bounds[:, 3].max(),
)
from shapely.geometry import box as shapely_box

combined_bounds_3979 = shp_transform(
    pyproj.Transformer.from_crs(project_crs, "EPSG:3979", always_xy=True).transform,
    shapely_box(*combined_bounds_project_crs),
).bounds
minx, miny, maxx, maxy = combined_bounds_3979

dem_full = rioxarray.open_rasterio(dem_dtm_url, masked=True)
dem_clip_3979 = dem_full.rio.clip_box(
    minx=minx - dem_clip_buffer_m, miny=miny - dem_clip_buffer_m,
    maxx=maxx + dem_clip_buffer_m, maxy=maxy + dem_clip_buffer_m,
)
dem = dem_clip_3979.rio.reproject(project_crs)
pixel_size = dem.rio.resolution()
print(f"DEM clip shape: {dem.shape}, CRS: {dem.rio.crs}, pixel size (m): {pixel_size}")
assert abs(abs(pixel_size[0]) - 1.0) < 0.1 and abs(abs(pixel_size[1]) - 1.0) < 0.1, (
    "expected ~1x1 m pixels after reprojection"
)

## Slope and aspect

Computed from the reprojected DEM array with a simple finite-difference gradient (no new package needed beyond numpy, already a geopandas dependency).

In [ ]:
import numpy as np

dem_arr = dem.values[0]
res_x, res_y = abs(pixel_size[0]), abs(pixel_size[1])
grad_y, grad_x = np.gradient(dem_arr, res_y, res_x)
slope_deg = np.degrees(np.arctan(np.sqrt(grad_x**2 + grad_y**2)))
slope_pct = np.tan(np.radians(slope_deg)) * 100
aspect_deg = np.degrees(np.arctan2(grad_y, -grad_x)) % 360

transform_affine = dem.rio.transform()
print(f"slope range (deg): {np.nanmin(slope_deg):.1f} to {np.nanmax(slope_deg):.1f}")

## Sample DEM elevation at a point

One small helper, reused by both the lift crosswalk and the per-run statistics below, so the same row/col lookup logic is not duplicated.

In [ ]:
def sample_at_point(array, x, y, affine):
    col, row = ~affine * (x, y)
    row, col = int(row), int(col)
    if 0 <= row < array.shape[0] and 0 <= col < array.shape[1]:
        return array[row, col]
    return np.nan


def sample_along_line(array, line, affine, n=20):
    pts = [line.interpolate(f, normalized=True) for f in np.linspace(0, 1, n)]
    return np.array([sample_at_point(array, p.x, p.y, affine) for p in pts])

## Lift crosswalk: DEM elevation and a proposed match to the plan

Bottom/top elevation and vertical rise per OSM lift, then two independent signals for a proposed plan map_ref match: the closest vertical rise (the only evidence available for the numbered map_refs, which carry no name), and, separately, the best name similarity (difflib.SequenceMatcher, a stdlib fuzzy match, not a hand-tuned rule) against the plan's three named map_refs. When the two signals disagree, both are reported rather than one silently overriding the other -- exactly the kind of case confirmed_by_jason exists for.

In [ ]:
import difflib

plan_lifts = (
    __import__("pandas").read_csv(f"{processed_dir}/03_lift_ccc_reproduction.csv")
    [["map_ref", "vert_rise_m"]].drop_duplicates(subset="map_ref")
)
named_map_refs = plan_lifts[~plan_lifts["map_ref"].str.match(r"^\d+$")]

crosswalk_rows = []
for _, lift in lifts_gdf.iterrows():
    coords = list(lift.geometry.coords)
    bottom_elev = sample_at_point(dem_arr, *coords[0], transform_affine)
    top_elev = sample_at_point(dem_arr, *coords[-1], transform_affine)
    dem_vert_rise = top_elev - bottom_elev

    diffs = (plan_lifts["vert_rise_m"] - dem_vert_rise).abs()
    best_idx = diffs.idxmin()
    best_match = plan_lifts.loc[best_idx, "map_ref"]
    best_delta = diffs.loc[best_idx]
    match_basis = f"vertical rise (delta={best_delta:.1f}m)"

    if lift["name"] and len(named_map_refs):
        name_scores = named_map_refs["map_ref"].apply(
            lambda ref: difflib.SequenceMatcher(None, lift["name"].lower(), ref.lower()).ratio()
        )
        top_name_idx = name_scores.idxmax()
        top_name_ref = named_map_refs.loc[top_name_idx, "map_ref"]
        top_name_score = name_scores.loc[top_name_idx]
        if top_name_score > 0.5 and top_name_ref != best_match:
            match_basis += (
                f"; name similarity also suggests {top_name_ref!r} "
                f"(ratio={top_name_score:.2f}), disagreeing with the vertical-rise match"
            )

    crosswalk_rows.append({
        "osm_id": lift["osm_id"], "osm_name": lift["name"], "aerialway_type": lift["aerialway"],
        "dem_bottom_elev_m": round(float(bottom_elev), 1), "dem_top_elev_m": round(float(top_elev), 1),
        "dem_vert_rise_m": round(float(dem_vert_rise), 1),
        "proposed_map_ref": best_match, "plan_vert_rise_m": plan_lifts.loc[best_idx, "vert_rise_m"],
        "vert_rise_delta_m": round(float(best_delta), 1),
        "match_basis": match_basis,
        "confirmed_by_jason": "",
    })

import pandas as pd
lift_crosswalk = pd.DataFrame(crosswalk_rows).sort_values("vert_rise_delta_m")
lift_crosswalk

## Terrain slope-class summary

For each of the two candidate threshold sets (C067, C069; neither sourced for groomed downhill terrain, hence testing both), classify every DEM pixel inside the AOI and report the area percentage per class. The Checks section below compares this against the master plan's own split (C032, read from the ledger, not retyped) for comparison, not validation -- the methods are different enough that a close match would be coincidental and a mismatch is not necessarily an error.

In [ ]:
from rasterio.features import geometry_mask

aoi_mask = ~geometry_mask(
    [aoi.geometry.iloc[0]], out_shape=dem_arr.shape, transform=transform_affine, invert=False
)
in_aoi = aoi_mask & ~np.isnan(slope_deg)

def classify(values, breaks):
    labels = np.digitize(values, breaks)
    return labels

terrain_summary_rows = []
for set_name, breaks in slope_threshold_sets.items():
    field = slope_pct if "pct" in set_name else slope_deg
    labels = classify(field[in_aoi], breaks)
    counts = pd.Series(labels).value_counts(normalize=True).sort_index() * 100
    for class_idx, pct in counts.items():
        terrain_summary_rows.append({
            "threshold_set": set_name, "class_index": int(class_idx), "area_pct": round(float(pct), 1),
        })
terrain_slope_summary = pd.DataFrame(terrain_summary_rows)
terrain_slope_summary

## Per-run statistics

Length and vertical drop from the run's own vertices (reprojected coordinates), slope sampled at 20 evenly-spaced points along each line, and the OSM-tagged difficulty kept alongside for comparison in the output table rather than collapsed into a single verdict here.

In [ ]:
run_rows = []
for _, run in runs_gdf.iterrows():
    length_m = run.geometry.length
    elevs = np.array([sample_at_point(dem_arr, x, y, transform_affine) for x, y in run.geometry.coords])
    vert_drop_m = np.nanmax(elevs) - np.nanmin(elevs) if len(elevs) else np.nan
    slopes_deg = sample_along_line(slope_deg, run.geometry, transform_affine)
    run_rows.append({
        "osm_id": run["osm_id"], "name": run["name"], "osm_difficulty": run["osm_difficulty"],
        "length_m": round(float(length_m), 1), "vert_drop_m": round(float(vert_drop_m), 1),
        "mean_slope_deg": round(float(np.nanmean(slopes_deg)), 1),
        "max_slope_deg": round(float(np.nanmax(slopes_deg)), 1),
    })
runs_stats = pd.DataFrame(run_rows)
runs_stats.head()

## Write outputs

GeoPackage for the two geometry layers (AOI, runs with their stats joined on), CSV for the tabular-only outputs.

In [ ]:
import os

os.makedirs(processed_dir, exist_ok=True)
aoi.to_file(f"{processed_dir}/03_aoi.gpkg", driver="GPKG")
lift_crosswalk.to_csv(f"{processed_dir}/03_lift_crosswalk.csv", index=False, encoding="utf-8")

runs_out = runs_gdf.merge(runs_stats, on="osm_id")[
    ["osm_id", "name_x", "osm_difficulty_x", "length_m", "vert_drop_m",
     "mean_slope_deg", "max_slope_deg", "geometry"]
].rename(columns={"name_x": "name", "osm_difficulty_x": "osm_difficulty"})
runs_out.to_file(f"{processed_dir}/03_runs.gpkg", driver="GPKG")

terrain_slope_summary.to_csv(f"{processed_dir}/03_terrain_slope_aspect_summary.csv", index=False, encoding="utf-8")
print("wrote 03_aoi.gpkg, 03_lift_crosswalk.csv, 03_runs.gpkg, 03_terrain_slope_aspect_summary.csv")

## Checks

AOI area against C032's independent figure, lift vertical rise against the plan for the closest proposed matches, and a CRS/units sanity check, per steps/03's stated Checks.

In [ ]:
assert ledger["claims"]["C032"]["source_id"] == "S034"
plan_terrain_area_ha = 1263  # from C032's own claim text; the master plan's stated terrain area
aoi_area_diff_pct = abs(aoi_area_ha - plan_terrain_area_ha) / plan_terrain_area_ha * 100
print(f"AOI area {aoi_area_ha:.1f} ha vs. plan's {plan_terrain_area_ha} ha: {aoi_area_diff_pct:.1f}% difference")
assert aoi_area_diff_pct < aoi_area_tolerance_pct, (
    f"AOI area diverges {aoi_area_diff_pct:.1f}% from the plan's own terrain area, "
    f"more than the {aoi_area_tolerance_pct}% tolerance"
)

close_matches = lift_crosswalk[lift_crosswalk["vert_rise_delta_m"] < lift_vert_rise_tolerance_m]
print(f"{len(close_matches)} of {len(lift_crosswalk)} lifts have a plan match within {lift_vert_rise_tolerance_m}m vertical rise")
assert len(close_matches) == len(lift_crosswalk), (
    f"{len(lift_crosswalk) - len(close_matches)} lift(s) have no plan match within "
    f"{lift_vert_rise_tolerance_m}m vertical rise -- inspect data/processed/03_lift_crosswalk.csv"
)

assert dem.rio.crs.to_epsg() == 26911
assert abs(abs(pixel_size[0]) - 1.0) < 0.1
print("checks passed")

## Map: AOI, lifts, and runs on a hillshade

One static map rendered inside the notebook, downsampled for display (the analysis above used the full 1 m resolution; a 6000x7000 pixel imshow would be needlessly slow to render).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

downsample = 5
dem_small = dem_arr[::downsample, ::downsample]
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(np.nan_to_num(dem_small, nan=np.nanmin(dem_small)), vert_exag=1.5)

fig, ax = plt.subplots(figsize=(9, 9))
extent = dem.rio.bounds()
ax.imshow(hillshade, cmap="gray", extent=(extent[0], extent[2], extent[1], extent[3]), origin="upper")
aoi.boundary.plot(ax=ax, color="yellow", linewidth=2, label="AOI")
lifts_gdf.plot(ax=ax, color="red", linewidth=1.5)
difficulty_colors = {"easy": "green", "intermediate": "blue", "advanced": "black", "expert": "black", "freeride": "orange"}
for diff, color in difficulty_colors.items():
    subset = runs_gdf[runs_gdf["osm_difficulty"] == diff]
    if len(subset):
        subset.plot(ax=ax, color=color, linewidth=0.6)
ax.set_title(f"RMR area of interest: AOI, {len(lifts_gdf)} lifts, {len(runs_gdf)} runs on HRDEM hillshade")
ax.set_xlabel(f"Easting ({project_crs})")
ax.set_ylabel("Northing")
plt.tight_layout()
plt.show()

## Versions

In [ ]:
import importlib.metadata

for pkg in ["pandas", "geopandas", "rasterio", "rioxarray", "shapely", "pyproj", "numpy", "matplotlib"]:
    print(pkg, importlib.metadata.version(pkg))